# 🌿 Leva-TTS — Quick Start

Generate your first Levantine-Arabic / English code-switching audio in a few cells.

**What this notebook does**
1. Sets up Python 3.10 (condacolab) and installs `leva-tts`
2. Loads the model (auto-downloads the fine-tuned checkpoint from HuggingFace)
3. Synthesizes speech with a built-in speaker, a code-switch sentence, zero-shot cloning, and streaming

👉 First set **Runtime ▸ Change runtime type ▸ T4 GPU**.

## Set up a Python 3.10 environment (condacolab)

> ⚠️ **Why:** Colab's default runtime is Python 3.12, but Coqui **`TTS`**
> (the XTTS engine) only supports Python **< 3.12**. We use **condacolab** to
> switch the runtime to **Python 3.10**, which `leva-tts` needs.

**Run the cell below once.** It will **automatically restart the kernel** — this
is expected. Wait for the restart, then **continue from the *next* cell**
(do **not** re-run this cell).

In [ ]:
# Installs Miniconda (Python 3.10) into Colab. The kernel auto-restarts after this —
# that is normal. Do NOT re-run this cell; just continue below once it reconnects.
!pip install -q condacolab
import condacolab
condacolab.install()        # ⏳ ~1 min, then the runtime restarts automatically

## Install Leva-TTS (on the Python 3.10 kernel)

After the restart above, the runtime is Python 3.10. Verify, then install the package.

In [ ]:
import condacolab; condacolab.check()      # confirms the 3.10 conda runtime is active
import sys; print('Python', sys.version.split()[0])

# Install Leva-TTS (+ audio system libs). ~3–5 min on first run.
!pip install -q "leva-tts"
!apt-get -qq install -y espeak-ng ffmpeg > /dev/null
print("✅ Leva-TTS installed on Python 3.10")

## Check the GPU

In [ ]:
# Verify a GPU runtime is active (Runtime ▸ Change runtime type ▸ T4 GPU)
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout or "⚠️ No GPU — switch the runtime to a GPU!")

## Load the model

First run downloads the checkpoint + 10 reference speakers (~2 GB) — give it a minute.

In [ ]:
from leva_tts import LevaTTS, SPEAKERS

tts = LevaTTS(device="cuda", preprocess_text=True, verbose=False)
print("Available speakers:", SPEAKERS)

## Synthesize with a built-in speaker

`synthesize(text, speaker, **gen_params)` → `(wav, sr)`. `speaker` must be one of `SPEAKERS`.

In [ ]:
from IPython.display import Audio, display
import soundfile as sf

wav, sr = tts.synthesize(
    "هَلَّق أنا عم أشتغل على the new project اللي حكيتلك عنه.",
    speaker="Badr",
    temperature=0.65,
)
sf.write("badr.wav", wav, sr)
print(f"{len(wav)/sr:.2f}s @ {sr} Hz")
display(Audio(wav, rate=sr))

## Try other speakers & a pure-Levantine line

In [ ]:
wav, sr = tts.synthesize("كيفك اليوم؟ إنت شو عم تعمل هَلَّق؟", speaker="Amina")
display(Audio(wav, rate=sr))

## Zero-shot voice cloning

Upload your own 3–10 s reference clip and clone the voice. Run the cell, pick a `.wav`/`.mp3`.

In [ ]:
from google.colab import files
up = files.upload()                 # choose a short reference clip
ref_path = next(iter(up))

wav, sr = tts.zero_shot_synthesize("مرحبا، هَلَّق عم نجرب الـ voice cloning.", ref_path)
display(Audio(wav, rate=sr))

## Streaming

`stream(...)` yields audio chunks as they're generated (useful for low-latency playback).

In [ ]:
import numpy as np
chunks = []
for ch in tts.stream("بِدِّي أحكيلك عن the new feature هَلَّق، رح تعجبك كتير.", speaker="Mona"):
    chunks.append(ch)
print(f"received {len(chunks)} chunks")
display(Audio(np.concatenate(chunks), rate=24000))

## 🎉 Done!

**Generation params** (optional on every method): `temperature`, `length_penalty`,
`repetition_penalty`, `top_k`, `top_p`, `speed`.

Next: [inference server](02_inference_server.ipynb) ·
[evaluation](03_evaluation.ipynb) · [Gradio app](04_gradio_app.ipynb).